**Integrantes**

- Castro Lozano, Ernesto Saniel
- Quispe Bernardo, Andrés


# 04 — Evaluation and Benchmarks

**Propósito:** evaluar recuperación y generación del sistema RAG sobre `medqa_test.parquet`, comparar las tres estrategias de chunking y los cuatro modos de retrieval, ejecutar RAGAS y consolidar benchmarks.

**Entradas requeridas**
- `medqa_test.parquet`
- `df_chunks_A_256.parquet`, `df_chunks_B_512.parquet`, `df_chunks_C_1024.parquet`
- `embeddings_A_256.npy`, `embeddings_B_512.npy`, `embeddings_C_1024.npy`

**Salidas**
- Métricas de grounding.
- Recall@k y Precision@k.
- Scores RAGAS.
- Tablas, análisis cualitativo y benchmark comparativo.

> Requiere GPU para Qwen2.5-7B-Instruct y un secreto `OPENAI_API_KEY` en Colab para la sección RAGAS.


In [ ]:
!pip install -q pandas pyarrow numpy sentence-transformers faiss-cpu rank_bm25 transformers accelerate "bitsandbytes>=0.46.1"

import pandas as pd
import numpy as np
from pathlib import Path


In [ ]:
strategies = ["A_256", "B_512", "C_1024"]
required_files = []
for s in strategies:
    required_files += [Path(f"df_chunks_{s}.parquet"), Path(f"embeddings_{s}.npy")]
required_files.append(Path("medqa_test.parquet"))

missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Faltan artefactos de los Notebooks 01/02: " + ", ".join(missing) +
        ". Ejecuta los notebooks previos o carga los archivos en el directorio actual."
    )

df_chunks = {}
embeddings = {}
for s in strategies:
    df_chunks[s] = pd.read_parquet(f"df_chunks_{s}.parquet")
    embeddings[s] = np.load(f"embeddings_{s}.npy")
    print(f"[{s}] chunks={len(df_chunks[s]):,} | embeddings={embeddings[s].shape}")

test = pd.read_parquet("medqa_test.parquet")
print(f"[TEST] {len(test):,} filas")


In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('all-mpnet-base-v2')
print(f"Modelo        : all-mpnet-base-v2")
print(f"Max tokens    : {embedding_model.max_seq_length}")
print(f"Dimensiones   : 768")

# 1. Reconstrucción del retrieval para evaluación


In [ ]:
import faiss
from rank_bm25 import BM25Okapi

# Índice Flat IP (similitud coseno) — baseline
indexes_flat = {}
for name, emb in embeddings.items():
    dim = emb.shape[1]

    emb_norm = emb.astype('float32').copy()
    faiss.normalize_L2(emb_norm)

    index = faiss.IndexFlatIP(dim)
    index.add(emb_norm)
    indexes_flat[name] = index
    print(f"[Flat IP] {name}: {index.ntotal:,} vectores indexados")

# Índice HNSW (búsqueda aproximada) — BONUS
indexes_hnsw = {}
for name, emb in embeddings.items():
    dim = emb.shape[1]

    emb_norm = emb.astype('float32').copy()
    faiss.normalize_L2(emb_norm)

    M = 32
    index = faiss.IndexHNSWFlat(dim, M, faiss.METRIC_INNER_PRODUCT)
    index.hnsw.efConstruction = 200
    index.add(emb_norm)
    indexes_hnsw[name] = index
    print(f"[HNSW IP] {name}: {index.ntotal:,} vectores indexados | M={M} | efConstruction=200")

# Índice BM25 — BONUS
bm25_indexes = {}
for name, df_c in df_chunks.items():
    tokenized = [text.lower().split() for text in df_c['chunk_text'].tolist()]
    bm25_indexes[name] = BM25Okapi(tokenized)
    print(f"[BM25] {name}: {len(tokenized):,} documentos indexados")

In [ ]:
from sentence_transformers import CrossEncoder

# Reranker Cross-Encoder — BONUS
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print(f"[Reranker] cross-encoder/ms-marco-MiniLM-L-6-v2 cargado")

# Flat / HNSW
# NOTA: se agregan document_url y question_focus al resultado para soportar
# la citación de fuente/artículo en la respuesta final (sección 7.2).
def retrieve(query, index, df_c, model, k=5):
    query_emb = model.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(query_emb)
    distances, indices = index.search(query_emb, k)
    results = df_c.iloc[indices[0]].copy()
    results['score'] = distances[0]
    return results[['chunk_text', 'question', 'document_source', 'document_url', 'question_focus', 'score']]

# Hybrid Search
def hybrid_retrieve(query, faiss_index, bm25_index, df_c, model, k=5, alpha=0.5):
    n = len(df_c)

    query_emb = model.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(query_emb)
    faiss_scores, faiss_indices = faiss_index.search(query_emb, n)
    faiss_score_arr = np.zeros(n)
    for idx, score in zip(faiss_indices[0], faiss_scores[0]):
        if idx < n:
            faiss_score_arr[idx] = score
    faiss_min, faiss_max = faiss_score_arr.min(), faiss_score_arr.max()
    if faiss_max > faiss_min:
        faiss_score_arr = (faiss_score_arr - faiss_min) / (faiss_max - faiss_min)

    tokenized_query = query.lower().split()
    bm25_scores = bm25_index.get_scores(tokenized_query)
    bm25_min, bm25_max = bm25_scores.min(), bm25_scores.max()
    if bm25_max > bm25_min:
        bm25_scores = (bm25_scores - bm25_min) / (bm25_max - bm25_min)

    combined = alpha * faiss_score_arr + (1 - alpha) * bm25_scores
    top_indices = np.argsort(combined)[::-1][:k]
    results = df_c.iloc[top_indices].copy()
    results['score_hybrid'] = combined[top_indices]
    results['score_faiss']  = faiss_score_arr[top_indices]
    results['score_bm25']   = bm25_scores[top_indices]
    return results[['chunk_text', 'question', 'document_source', 'document_url', 'question_focus', 'score_hybrid', 'score_faiss', 'score_bm25']]

# Reranker
def retrieve_with_reranker(query, faiss_index, df_c, model, k_retrieve=20, k_final=5):
    query_emb = model.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(query_emb)
    distances, indices = faiss_index.search(query_emb, k_retrieve)
    candidates = df_c.iloc[indices[0]].copy()
    candidates['faiss_score'] = distances[0]
    pairs = [[query, chunk] for chunk in candidates['chunk_text'].tolist()]
    rerank_scores = reranker.predict(pairs)
    candidates['rerank_score'] = rerank_scores
    candidates = candidates.sort_values('rerank_score', ascending=False).head(k_final)
    return candidates[['chunk_text', 'question', 'document_source', 'document_url', 'question_focus', 'faiss_score', 'rerank_score']]

# 2. Carga del LLM y pipeline base


In [ ]:
import torch
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True
)

llm.eval()
print(f"Modelo cargado: {model_name}")
print(f"VRAM usada: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

In [ ]:
def ask_llm(context, question):
    messages = [
        {"role": "system", "content": "Answer ONLY using provided context. If the answer is not in the context, say 'I don't know'."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion:\n{question}"}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)

    with torch.inference_mode():
        out = llm.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,       # respuestas deterministas
            temperature=1.0,
            repetition_penalty=1.1 # evita repeticiones
        )

    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)


def rag_pipeline(question, strategy, mode, k=5):
    """
    strategy: 'A_256', 'B_512', 'C_1024'
    mode: 'flat', 'hnsw', 'hybrid', 'reranker'
    """
    df_c = df_chunks[strategy]

    if mode == 'flat':
        results = retrieve(question, indexes_flat[strategy], df_c, embedding_model, k=k)
    elif mode == 'hnsw':
        results = retrieve(question, indexes_hnsw[strategy], df_c, embedding_model, k=k)
    elif mode == 'hybrid':
        results = hybrid_retrieve(question, indexes_flat[strategy], bm25_indexes[strategy], df_c, embedding_model, k=k)
    elif mode == 'reranker':
        results = retrieve_with_reranker(question, indexes_flat[strategy], df_c, embedding_model)

    context = "\n\n".join(results['chunk_text'].tolist())
    answer  = ask_llm(context, question)
    return answer, results

In [ ]:
def format_citations(results):
    """
    Construye el bloque de citas (fuente + artículo) a partir de los chunks
    recuperados, deduplicando por document_url para no repetir el mismo
    artículo varias veces si aportó más de un chunk.
    """
    seen = set()
    citations = []
    for _, r in results.iterrows():
        key = r['document_url']
        if key in seen:
            continue
        seen.add(key)
        titulo = r['question_focus'] if pd.notna(r['question_focus']) else "(sin título)"
        citations.append(f"[{r['document_source']}] {titulo} — {r['document_url']}")
    return citations


def rag_pipeline_with_citations(question, strategy, mode, k=5):
    """
    Igual que rag_pipeline(), pero además devuelve el bloque de citas
    (fuente + artículo) listo para mostrar junto a la respuesta.
    """
    answer, results = rag_pipeline(question, strategy=strategy, mode=mode, k=k)
    citations = format_citations(results)
    return answer, results, citations


# 3. Funciones de grounding


In [ ]:
import re

stopwords_en = {
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're",
    "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he',
    'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's",
    'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what',
    'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am',
    'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had',
    'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but',
    'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for',
    'with', 'about', 'against', 'between', 'into', 'through', 'during',
    'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in',
    'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once'
}

def hallucination_guard(answer, context):
    ans_words = [word for word in re.findall(r'\w+', answer.lower()) if word not in stopwords_en]
    ctx_words = set(re.findall(r'\w+', context.lower()))

    if not ans_words:
        return 0.0

    hits = sum(1 for word in ans_words if word in ctx_words)
    ratio = hits / len(ans_words)
    return ratio

def grounding_words(answer, retrieved_chunks):
    ctx   = " ".join(retrieved_chunks)
    score = hallucination_guard(answer, ctx)
    label = "✅ Bien fundamentada" if score >= 0.8 else "⚠️ Parcialmente fundamentada" if score >= 0.5 else "❌ Posible alucinación"
    ans_words = [w for w in re.findall(r'\w+', answer.lower()) if w not in stopwords_en]
    ctx_words = set(re.findall(r'\w+', ctx.lower()))
    return {
        'grounding_score': round(score, 3),
        'label'          : label,
        'total_words'    : len(ans_words),
        'words_matched'  : sum(1 for w in ans_words if w in ctx_words)
    }

def grounding_sentences(answer, retrieved_chunks):
    ctx       = " ".join(retrieved_chunks).lower()
    ctx_words = set(re.findall(r'\w+', ctx))
    sentences = [s.strip() for s in re.split(r'[.!?]', answer) if len(s.strip()) > 10]

    if not sentences:
        return {'grounding_score': 0.0, 'label': '❌ Sin frases', 'total_sentences': 0, 'supported': 0}

    supported = 0
    for sentence in sentences:
        words = [w for w in re.findall(r'\w+', sentence.lower()) if w not in stopwords_en]
        if not words:
            continue
        hits = sum(1 for w in words if w in ctx_words)
        if hits / len(words) >= 0.6:
            supported += 1

    score = supported / len(sentences)
    label = "✅ Bien fundamentada" if score >= 0.8 else "⚠️ Parcialmente fundamentada" if score >= 0.5 else "❌ Posible alucinación"
    return {
        'grounding_score'  : round(score, 3),
        'label'            : label,
        'total_sentences'  : len(sentences),
        'supported'        : supported
    }

In [ ]:
# Aplicar ambos a todas las variantes
question = "What are the symptoms of Bell's palsy?"
strategies = ['A_256', 'B_512', 'C_1024']
modes      = ['flat', 'hnsw', 'hybrid', 'reranker']

results_grounding = []

for strategy in strategies:
    for mode in modes:
        answer, chunks = rag_pipeline(question, strategy=strategy, mode=mode)
        chunk_texts    = chunks['chunk_text'].tolist()

        r_words = grounding_words(answer, chunk_texts)
        r_sents = grounding_sentences(answer, chunk_texts)

        results_grounding.append({
            'strategy'            : strategy,
            'mode'                : mode,
            'score_words'         : r_words['grounding_score'],
            'label_words'         : r_words['label'],
            'words_matched'       : r_words['words_matched'],
            'total_words'         : r_words['total_words'],
            'score_sentences'     : r_sents['grounding_score'],
            'label_sentences'     : r_sents['label'],
            'sentences_supported' : r_sents['supported'],
            'total_sentences'     : r_sents['total_sentences'],
            'answer'              : answer[:150] + '...' if len(answer) > 150 else answer
        })

        print(f"[{strategy}][{mode}] palabras={r_words['grounding_score']} | frases={r_sents['grounding_score']}")

df_grounding = pd.DataFrame(results_grounding)
display(df_grounding[['strategy', 'mode', 'score_words', 'label_words', 'score_sentences', 'label_sentences']])

In [ ]:
questions = [
    "What are the symptoms of Bell's palsy?",
    "How is spinocerebellar ataxia inherited?",
    "What are the treatments for Wernicke-Korsakoff syndrome?"
]

strategies = ['A_256', 'B_512', 'C_1024']
modes      = ['flat', 'hnsw', 'hybrid', 'reranker']

results_grounding = []

for question in questions:
    for strategy in strategies:
        for mode in modes:
            answer, chunks = rag_pipeline(question, strategy=strategy, mode=mode)
            chunk_texts    = chunks['chunk_text'].tolist()

            r_words = grounding_words(answer, chunk_texts)
            r_sents = grounding_sentences(answer, chunk_texts)

            results_grounding.append({
                'question'            : question[:60] + '...',
                'strategy'            : strategy,
                'mode'                : mode,
                'score_words'         : r_words['grounding_score'],
                'label_words'         : r_words['label'],
                'words_matched'       : r_words['words_matched'],
                'total_words'         : r_words['total_words'],
                'score_sentences'     : r_sents['grounding_score'],
                'label_sentences'     : r_sents['label'],
                'sentences_supported' : r_sents['supported'],
                'total_sentences'     : r_sents['total_sentences'],
                'answer'              : answer[:150] + '...' if len(answer) > 150 else answer
            })

            print(f"[{question[:40]}][{strategy}][{mode}] palabras={r_words['grounding_score']} | frases={r_sents['grounding_score']}")

df_grounding = pd.DataFrame(results_grounding)

# Mostrar por pregunta
for q in questions:
    print(f"\n{'='*60}")
    print(f"Pregunta: {q}")
    print(f"{'='*60}")
    mask = df_grounding['question'] == q[:60] + '...'
    display(df_grounding[mask][['strategy', 'mode', 'score_words', 'label_words', 'score_sentences', 'label_sentences']])

# Resumen promedio por estrategia y modo
print(f"\n{'='*60}")
print("Promedio de grounding scores por estrategia y modo:")
print(f"{'='*60}")
display(df_grounding.groupby(['strategy', 'mode'])[['score_words', 'score_sentences']].mean().round(3))

**Interpretación de resultados de grounding (3 preguntas, 12 combinaciones):**

Los resultados sobre 3 preguntas distintas revelan patrones más complejos
que los observados con una sola pregunta. La relación "chunk más grande =
mejor grounding" no se mantiene de forma consistente.

**Por modo de retrieval (promedio de las 3 preguntas):**

- **flat**: el más consistente en A_256 (0.825/0.833) y B_512 (0.828/0.833).
  Logra scores perfectos en Wernicke-Korsakoff para B_512 (1.0/1.0) y en
  spinocerebellar para A_256 (1.0/1.0). Es el modo más estable entre preguntas
  y estrategias.

- **hnsw**: el peor modo en promedio — 0.545/0.500 en A_256 y 0.535/0.500
  en B_512. El caso más crítico es Wernicke-Korsakoff: 0.071/0.0 en A_256
  y 0.120/0.0 en B_512 — scores de alucinación total. Sin embargo, en
  spinocerebellar con A_256 logra 1.0/1.0, igual que flat. Esto confirma
  que HNSW es inconsistente: funciona bien para términos comunes pero falla
  con términos médicos compuestos poco frecuentes.

- **hybrid**: muy inconsistente entre preguntas. Perfecto en Bell's palsy
  con B_512 (1.0/1.0) y en spinocerebellar con B_512 (1.0/1.0), pero
  colapsa en Wernicke-Korsakoff con B_512 (0.238/0.0) y A_256 (0.536/0.333).
  Su mejor resultado global es C_1024 + hybrid (0.980/1.000).

- **reranker**: razonable en Bell's palsy (0.878/1.0 en A_256, 0.969/1.0 en B_512)
  y spinocerebellar (0.815/1.0 en A_256, 0.944/1.0 en B_512), pero falla
  en Wernicke-Korsakoff con B_512 (0.476/0.0) y A_256 (0.500/0.4).
  Promedio general de 0.731-0.796 según estrategia.

**Por estrategia:**

- **A_256**: extremadamente variable. Excelente en spinocerebellar con
  flat/hnsw (1.0/1.0) pero débil en Bell's palsy con flat (0.519/0.5).
  El peor resultado de toda la evaluación ocurre aquí: hnsw en
  Wernicke-Korsakoff (0.071/0.0). Promedio flat 0.825/0.833
  pero ocultando alta varianza entre preguntas.

- **B_512**: flat destaca como el modo más robusto — scores perfectos en
  Bell's palsy (1.0 hybrid, 0.969 reranker) y Wernicke-Korsakoff (1.0/1.0).
  Sin embargo hnsw colapsa en Wernicke-Korsakoff (0.120/0.0) — diferencia
  de 0.88 puntos con flat para la misma estrategia, lo cual es alarmante.

- **C_1024**: hybrid es claramente el mejor modo (1.0/1.0 en las 3 preguntas,
  promedio 0.980/1.000). flat y hnsw son idénticos (0.788/0.726) como se
  esperaba de una búsqueda aproximada bien calibrada. El reranker baja
  inesperadamente en spinocerebellar (0.758/0.5).

**Hallazgo más importante:**

El caso de Wernicke-Korsakoff con hnsw expone una debilidad real del sistema:
scores de 0.071 y 0.120 en palabras, 0.0 en frases — alucinación total.
HNSW devuelve vecinos aproximados incorrectos para este término médico
compuesto, y el LLM genera una respuesta sin ningún respaldo en el contexto.
Esto es exactamente el tipo de fallo que el grounding está diseñado para detectar.

**Conclusión:**

La combinación **C_1024 + hybrid** obtiene el mejor grounding promedio
(0.980/1.000) y es consistente en las 3 preguntas. **B_512 + flat** es
la opción más estable y robusta (0.828/0.833) sin los riesgos de hybrid.
**hnsw debe descartarse** para este corpus dado su comportamiento
catastrófico con términos médicos específicos como Wernicke-Korsakoff.

# 9.Evaluación

## 9.1 Evaluación de retrieval — Recall@k y Precision@k

In [ ]:
# Muestra estratificada por question_type
test_sample = (
    test
    .groupby('question_type', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), max(1, int(200 * len(x) / len(test)))), random_state=42))
    .reset_index(drop=True)
)

print(f"Evaluando sobre {len(test_sample)} preguntas")
print(f"\nDistribución por question_type:")
print(test_sample['question_type'].value_counts())

In [ ]:
# Recall@k y Precision@k
from sklearn.metrics.pairwise import cosine_similarity

def evaluate_retrieval(test_df, strategy, mode, k=5, threshold=0.5):
    hits        = 0
    precisions  = []

    for _, row in test_df.iterrows():
        question       = row['question']
        expected_answer = row['answer']

        # Recuperar top-k chunks
        df_c = df_chunks[strategy]
        if mode == 'flat':
            results = retrieve(question, indexes_flat[strategy], df_c, embedding_model, k=k)
        elif mode == 'hnsw':
            results = retrieve(question, indexes_hnsw[strategy], df_c, embedding_model, k=k)
        elif mode == 'hybrid':
            results = hybrid_retrieve(question, indexes_flat[strategy], bm25_indexes[strategy], df_c, embedding_model, k=k)
        elif mode == 'reranker':
            results = retrieve_with_reranker(question, indexes_flat[strategy], df_c, embedding_model)

        # Embed respuesta esperada y chunks recuperados
        expected_emb = embedding_model.encode([expected_answer], convert_to_numpy=True)
        chunks_emb   = embedding_model.encode(results['chunk_text'].tolist(), convert_to_numpy=True)

        # Similitud coseno entre respuesta esperada y cada chunk
        sims = cosine_similarity(expected_emb, chunks_emb)[0]

        # Recall@k: ¿algún chunk supera el umbral?
        relevant = sims >= threshold
        if relevant.any():
            hits += 1

        # Precision@k: ¿cuántos de los k chunks son relevantes?
        precisions.append(relevant.sum() / k)

    recall_at_k    = hits / len(test_df)
    precision_at_k = sum(precisions) / len(test_df)

    return {
        'strategy'      : strategy,
        'mode'          : mode,
        'recall@k'      : round(recall_at_k, 3),
        'precision@k'   : round(precision_at_k, 3),
        'k'             : k,
        'threshold'     : threshold
    }

In [ ]:
# Evaluar todas las combinaciones
strategies = ['A_256', 'B_512', 'C_1024']
modes      = ['flat', 'hnsw', 'hybrid', 'reranker']

results_eval = []
for strategy in strategies:
    for mode in modes:
        print(f"Evaluando [{strategy}][{mode}]...")
        result = evaluate_retrieval(test_sample, strategy, mode, k=5, threshold=0.5)
        results_eval.append(result)
        print(f"  Recall@5={result['recall@k']} | Precision@5={result['precision@k']}")

df_eval = pd.DataFrame(results_eval)
print("\nResultados finales:")
display(df_eval.sort_values('recall@k', ascending=False))

**Evaluación completa del sistema RAG — Recall@5, Precision@5 y Grounding**

Combinando las métricas de retrieval (sobre 200 preguntas estratificadas)
con las de grounding (sobre 3 preguntas representativas), emergen patrones
que no serían visibles analizando cada métrica por separado.

---

**Hallazgo principal: alto Recall no implica alto Grounding**

El caso más claro es A_256 + flat: mejor Recall@5 del sistema (0.979)
pero grounding moderado (0.825/0.833). El sistema encuentra el chunk
correcto casi siempre, pero los chunks son tan pequeños que el LLM
no tiene suficiente contexto para generar una respuesta bien fundamentada.

El caso opuesto es C_1024 + hybrid: Recall@5 de 0.954 (no el mejor)
pero grounding perfecto (0.980/1.000). Sin embargo, tiene un punto débil
claro: Precision@5 de 0.785 — la más baja entre las combinaciones con
C_1024. Esto significa que aunque recupera contexto suficiente para
fundamentar la respuesta, incluye más chunks irrelevantes entre los
top-5 que flat o reranker. BM25 recupera chunks léxicamente similares
que no siempre son semánticamente relevantes.

---

**Por modo de retrieval:**

- **flat**: mejor modo en Recall@5 para A_256 (0.979) y C_1024 (0.959),
  y mejor en Grounding para A_256 (0.825/0.833) y B_512 (0.828/0.833).
  Es el modo más consistente en ambas métricas.

- **reranker**: segundo en Recall@5 (0.964-0.969) y grounding razonable
  (0.731-0.882). El cross-encoder mejora precisión sin sacrificar recall,
  pero no compensa el grounding bajo de chunks pequeños.

- **hybrid**: tercer lugar en Recall@5 (0.928-0.954) pero mejor grounding
  para C_1024 (0.980/1.000). Su punto débil es Precision@5 — 0.785 en
  C_1024, 0.798 en B_512 y 0.764 en A_256, las más bajas de sus
  respectivas estrategias. BM25 aporta precisión léxica que mejora
  la fundamentación pero introduce ruido en los resultados recuperados.

- **hnsw**: último en Recall@5 (0.912-0.933) y peor en grounding
  (0.535-0.788). El único modo que falla en ambas métricas simultáneamente.
  No recomendable para este corpus.

---

**Por estrategia:**

- **A_256**: mejor Recall@5 con flat (0.979) pero grounding inconsistente.
  Útil si el objetivo es encontrar información, no si se necesita
  contexto suficiente para el LLM.

- **B_512**: el balance más sólido — Recall@5 de 0.954 con flat,
  Precision@5 de 0.856 (mejor del sistema), grounding de 0.828/0.833.
  No lidera en ninguna métrica individual pero es consistente en todas.

- **C_1024**: hybrid es su mejor modo en grounding (0.980/1.000) pero
  al costo de la Precision@5 más baja (0.785). flat equilibra mejor
  las tres métricas — Recall 0.959, Precision 0.852, grounding 0.788/0.726.

---

**Resumen consolidado:**

| Métrica | Mejor combinación | Valor |
|---|---|---|
| Recall@5 | A_256 + flat | 0.979 |
| Precision@5 | B_512 + flat | 0.856 |
| Grounding words | C_1024 + hybrid | 0.980 |
| Grounding frases | C_1024 + hybrid | 1.000 |
| **Balance general** | **B_512 + flat** | — |

**B_512 + flat** es la combinación más robusta considerando todas las
métricas: Recall@5 de 0.954, Precision@5 de 0.856 y grounding de
0.828/0.833. A diferencia de C_1024 + hybrid — que lidera en grounding
pero sacrifica Precision@5 (0.785) — B_512 + flat mantiene rendimiento
alto y equilibrado sin puntos débiles críticos, y sin los fallos
catastróficos de hnsw en términos médicos específicos.

## 9.2 Evaluación con métricas RAGAS

La evaluación anterior (Recall@k, Precision@k, grounding) se hizo sobre una
muestra grande del propio split de test de MedQuAD, donde **todas** las
preguntas tienen garantizada una respuesta en el corpus. Eso es útil para
medir qué tan bien recupera el sistema, pero no permite verificar que el
sistema reconozca cuándo una pregunta **no** puede responderse con el corpus
disponible.

Para eso se construye un set de evaluación más pequeño y curado a mano
(13 preguntas), que incluye 3 **preguntas trampa** — fuera del
dominio médico del corpus — y sobre el cual se calculan tres métricas
del framework **RAGAS**:

- **Faithfulness**: proporción de afirmaciones de la respuesta que pueden
  inferirse del contexto recuperado. Mide alucinación.
- **Answer Relevancy**: qué tan pertinente es la respuesta generada respecto
  a la pregunta original. Se calcula generando preguntas sintéticas a partir
  de la respuesta y comparándolas por similitud coseno (embeddings) con la
  pregunta real; penaliza respuestas incompletas, redundantes o que se
  desvían del tema.
- **Context Utilization**: versión *reference-free* de `context_precision`.
  Evalúa, usando pregunta + respuesta + contextos recuperados, si los
  fragmentos realmente relevantes para responder quedan mejor rankeados
  dentro del contexto. No requiere una respuesta de referencia (`ground
  truth`), lo cual es conveniente porque el set curado no la tiene.

Como juez se usa `gpt-4o-mini` de OpenAI (vía `langchain_openai.ChatOpenAI`)
junto con el modelo de embeddings `text-embedding-3-small`, consumiendo la
API de OpenAI a través de `OPENAI_API_KEY`.

**Set de evaluación curado (13 preguntas, 3 trampa)**

Las preguntas 1-10 son representativas del dominio (distintos `question_type`:
información, síntomas, tratamiento, causas, complicaciones, frecuencia). Las
preguntas 11-13 son preguntas trampa: están fuera del dominio cubierto por
MedQuAD (no son preguntas médicas, o son medicina veterinaria/dental que el
corpus no cubre), y el sistema debería responder "I don't know" en lugar de
inventar una respuesta.


In [ ]:
# Set de evaluación curado: 13 preguntas, 10 representativas del dominio + 3 trampa
eval_set = [
    # --- Representativas del dominio (dentro del corpus) ---
    {"question": "What are the symptoms of Bell's palsy?", "is_trap": False},
    {"question": "What are the treatments for migraine?", "is_trap": False},
    {"question": "What causes Pleurisy and Other Pleural Disorders?", "is_trap": False},
    {"question": "What are the complications of Renal Artery Stenosis?", "is_trap": False},
    {"question": "How many people are affected by spastic paraplegia type 4?", "is_trap": False},
    {"question": "What is (are) keratoderma with woolly hair?", "is_trap": False},
    {"question": "What are the symptoms of Wernicke-Korsakoff syndrome?", "is_trap": False},
    {"question": "Is spinocerebellar ataxia inherited?", "is_trap": False},
    {"question": "What are the treatments for viral hepatitis?", "is_trap": False},
    {"question": "What causes diabetic retinopathy?", "is_trap": False},
    # --- Preguntas trampa (fuera del corpus médico humano) ---
    {"question": "What is the best diet for a dog with kidney disease?", "is_trap": True},
    {"question": "How do I change the oil in my car engine?", "is_trap": True},
    {"question": "What is the capital city of Peru?", "is_trap": True},
]

df_eval_set = pd.DataFrame(eval_set)
print(f"Total preguntas: {len(df_eval_set)} | Trampa: {df_eval_set['is_trap'].sum()}")
display(df_eval_set)


**Generación de respuestas y contextos sobre el set de evaluación**

Antes de poder calcular las métricas RAGAS se necesita, para cada pregunta
del set: la respuesta generada por el sistema y los chunks de contexto que
se usaron para generarla. Se reutiliza `rag_pipeline()` con la mejor
combinación encontrada en la sección 9.1 (`B_512` + `flat`, ver sección 10).


In [ ]:
from datasets import Dataset

# Generar respuesta + contexto para cada pregunta del set de evaluación,
# usando la mejor combinación (B_512 + flat, ver sección 10) y pasando por
# los guardrails (rag_pipeline_safe), igual que cualquier pregunta real de
# un usuario.
ragas_rows = []

for _, row in df_eval_set.iterrows():
    question = row['question']
    answer, results = rag_pipeline(question, strategy='B_512', mode='flat', k=5)

    # Si el guardrail bloqueó la pregunta, results es None — no hay contexto
    # que pasarle a RAGAS, así que se usa una lista vacía.
    contexts = results['chunk_text'].tolist() if results is not None else []

    ragas_rows.append({
        'question'    : question,
        'answer'      : answer,
        'contexts'    : contexts,
        'is_trap'     : row['is_trap'],
    })

    print(f"[{'TRAMPA' if row['is_trap'] else 'normal'}] {question}")
    print(f"  -> {answer[:500]}")
    print()

df_ragas_input = pd.DataFrame(ragas_rows)
ragas_dataset = Dataset.from_pandas(df_ragas_input[['question', 'answer', 'contexts']])
print(ragas_dataset)

**LLM externo (OpenAI) como juez de RAGAS**

`ragas` espera un LLM compatible con la interfaz de LangChain. Inicialmente
se intentó envolver `Qwen2.5-7B-Instruct` (el LLM local del pipeline, ya
cargado desde la sección 7) en un `pipeline` de `transformers` +
`HuggingFacePipeline` + `LangchainLLMWrapper`, para no depender de una API
externa de pago.

Sin embargo, en la práctica esto resultó inviable: cada "job" de RAGAS
implica varias llamadas encadenadas al LLM (descomponer la respuesta en
afirmaciones, verificar cada una contra el contexto, generar preguntas
inversas, etc.), y un modelo de 7B cuantizado corriendo localmente en la
GPU de Colab es demasiado lento para sostener ese volumen de llamadas —
esto generaba `TimeoutError` recurrentes y errores de parseo de JSON
(el modelo no seguía el formato estricto que RAGAS necesita), incluso
ajustando el `timeout` y los reintentos.

Por eso se usa **GPT-4o-mini** (vía API de OpenAI) únicamente como juez de
las métricas RAGAS — es rápido, económico, y sigue instrucciones de formato
de forma mucho más confiable. La API key se gestiona mediante los Secretos
de Colab, como se indica en las recomendaciones del enunciado.

Es importante notar que esto **no afecta al sistema RAG en sí**: las
respuestas que el usuario recibe siguen siendo generadas por
`Qwen2.5-7B-Instruct` local (sección 7); GPT-4o-mini solo evalúa esas
respuestas ya generadas, no participa en absoluto del pipeline de
producción del asistente.

In [ ]:
!pip install -q -U "ragas==0.1.21" "langchain-openai>=0.1.8,<0.2" "langchain>=0.2.2,<0.3" "langchain-core>=0.2.4,<0.3" "langchain-community>=0.2.4,<0.3"

import sys
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('pydantic') or mod_name.startswith('langchain'):
        del sys.modules[mod_name]

import os
from google.colab import userdata
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

openai_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
ragas_llm = LangchainLLMWrapper(openai_llm)

openai_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
ragas_embeddings = LangchainEmbeddingsWrapper(openai_embeddings)

print("LLM y embeddings de OpenAI envueltos para RAGAS.")


> ⚠️ Nota de reproducibilidad: usar un LLM local de 7B cuantizado como juez
> (en vez de un modelo grande vía API) puede producir scores más ruidosos
> que con GPT-4 u otro juez de mayor capacidad — es una limitación conocida
> y se comenta en el análisis de resultados (sección 10).


In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_utilization
from ragas.run_config import RunConfig

run_config = RunConfig(
    timeout=60,
    max_retries=3,
    max_wait=30,
    max_workers=3,
)

ragas_results = evaluate(
    dataset=ragas_dataset,
    metrics=[faithfulness, answer_relevancy, context_utilization],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    run_config=run_config,
)

df_ragas_scores = ragas_results.to_pandas()
df_ragas_scores['is_trap'] = df_ragas_input['is_trap'].values

print("Scores RAGAS por pregunta:")
display(df_ragas_scores[['question', 'is_trap', 'faithfulness', 'answer_relevancy', 'context_utilization']])

print("\nPromedios generales:")
print(df_ragas_scores[['faithfulness', 'answer_relevancy', 'context_utilization']].mean())

print("\nPromedios — preguntas normales vs trampa:")
print(df_ragas_scores.groupby('is_trap')[['faithfulness', 'answer_relevancy', 'context_utilization']].mean())

Las preguntas normales muestran faithfulness (0.86) y context_utilization (0.96) altos, indicando que el sistema fundamenta bien sus respuestas en el corpus recuperado. El answer_relevancy más moderado (0.82) sugiere margen de mejora en precisión de las respuestas. Las preguntas trampa muestran una caída drástica en las tres métricas (0.17 / 0.00 / 0.08), confirmando que el sistema reconoce correctamente cuándo una pregunta está fuera de su dominio en lugar de alucinar — comportamiento deseado, no una falla.

El paper original de RAGAS (Es et al., 2023, arXiv:2309.15217) definió conceptualmente 'Context Relevance' como una de sus tres métricas centrales. En las versiones actuales de la librería, ese concepto se implementa bajo el nombre context_precision (cuando hay ground_truth) o context_utilization (cuando no lo hay, que es nuestro caso, ya que el PDF especifica evaluación sin respuestas de referencia). Ambas calculan la misma fórmula de precisión promedio sobre verificaciones LLM de relevancia del contexto — solo cambia si se compara contra una respuesta humana o contra la respuesta generada por el sistema

**Interpretación de resultados RAGAS**

| Métrica | Promedio (normales) | Promedio (trampa) | Lectura |
|---|---|---|---|
| **Faithfulness** | 0.86 | 0.17 | Consistente con el grounding por palabras/frases de la sección 8. El LLM-juez (GPT-4o-mini) es más estricto que esa heurística: penaliza afirmaciones médicamente correctas pero no inferibles *literalmente* del contexto (casos #3, #7, #9, todos entre 0.57–0.60). |
| **Answer Relevance** | 0.82 | 0.00 | Buen desempeño general, con una excepción notable: pregunta #7 ("Is spinocerebellar ataxia inherited?") tiene Faithfulness 0.57 y Context Utilization 0.81, pero Answer Relevance **0.00** — respuesta fundamentada pero que no atiende directamente lo preguntado (rodeos/información de más, tal como advierte la guía). |
| **Context Utilization** | 0.94 | 0.08 | Muy alto y estable — el top-k (k=5, B_512, flat) trae mayormente chunks relevantes. Única caída: pregunta #8 (hepatitis viral) en 0.80. |
| **Preguntas trampa** | — | — | Caída clara y consistente en las 3 métricas, confirmando que el sistema reconoce preguntas fuera de dominio en vez de alucinar — comportamiento correcto, no una falla. Único caso atípico: pregunta #11 ("car engine oil") con Faithfulness 0.5 en vez de 0, sugiriendo que el rechazo no fue 100% limpio. |

**Conclusión:** el sistema evita alucinaciones ante preguntas fuera de dominio (objetivo central del grounding) y mantiene desempeño sólido dentro del corpus. Context Utilization es la dimensión más fuerte (0.94); Answer Relevance es la de mayor margen de mejora (0.82), indicando que el cuello de botella no está en la recuperación sino en qué tan preciso es el LLM al responder exactamente lo preguntado.

# 10.Análisis de resultados

A lo largo de la evaluación se compararon 12 combinaciones (3 estrategias
de chunking × 4 modos de retrieval) usando tres métricas: Recall@5,
Precision@5 y Grounding ratio. Los resultados revelan que no existe una
combinación perfecta — cada una presenta fortalezas y debilidades según
la métrica evaluada.

---

**Principales insights:**

1. **Alto Recall no implica alto Grounding.** A_256 + flat lidera en
   Recall@5 (0.979) pero tiene grounding moderado (0.825/0.833). Chunks
   pequeños encuentran la información pero no proporcionan contexto
   suficiente al LLM para fundamentar la respuesta.

2. **HNSW es inconsistente en corpus médico.** A pesar de ser más eficiente
   computacionalmente, falla de forma catastrófica con términos médicos
   compuestos poco frecuentes (Wernicke-Korsakoff: 0.071/0.0 en grounding).
   Es el único modo que falla simultáneamente en retrieval y grounding.

3. **Hybrid mejora grounding pero sacrifica precisión.** C_1024 + hybrid
   alcanza grounding perfecto (0.980/1.000) pero tiene la Precision@5
   más baja de su estrategia (0.785) — recupera más ruido junto con
   la información relevante.

4. **El tamaño de chunk afecta de forma distinta cada métrica.** Chunks
   pequeños (A_256) maximizan Recall, chunks grandes (C_1024) maximizan
   Grounding, chunks medianos (B_512) equilibran ambos.

---

**Top 3 combinaciones:**

| Posición | Combinación | Recall@5 | Precision@5 | Grounding | Motivo |
|---|---|---|---|---|---|
| 🥇 | B_512 + flat | 0.954 | 0.856 | 0.828/0.833 | Mejor balance entre las 3 métricas sin puntos débiles |
| 🥈 | C_1024 + hybrid | 0.954 | 0.785 | 0.980/1.000 | Mejor grounding del sistema pero precisión baja |
| 🥉 | A_256 + flat | 0.979 | 0.838 | 0.825/0.833 | Mejor recall pero grounding inconsistente entre preguntas |

---

**¿Por qué gana B_512 + flat?**

La elección del ganador se basa en tres criterios:

- **Consistencia entre métricas**: es la única combinación que mantiene
  rendimiento alto en Recall@5 (0.954), Precision@5 (0.856) y Grounding
  (0.828/0.833) simultáneamente, sin sacrificar ninguna métrica por otra.

- **Robustez entre tipos de pregunta**: no presenta fallos catastróficos
  en ninguna de las 3 preguntas evaluadas en grounding, a diferencia de
  hnsw que colapsa con términos específicos.

- **Adecuación al dominio médico**: en un sistema de QA médico es crítico
  que las respuestas estén bien fundamentadas en el contexto recuperado.
  B_512 proporciona chunks con suficiente contexto para el LLM sin el
  ruido que introduce hybrid o la fragmentación de A_256.

La búsqueda exacta (flat) sobre embeddings normalizados con similitud
coseno resulta ser la estrategia más confiable para este corpus, sin
la inestabilidad de HNSW ni la penalización en precisión del hybrid search.

# 11.Demo análisis cualitativo

In [ ]:
def demo_question(question, strategy='B_512', mode='flat', k=5):
    answer, chunks  = rag_pipeline(question, strategy=strategy, mode=mode, k=k)
    citations       = format_citations(chunks)
    chunk_texts     = chunks['chunk_text'].tolist()
    r_words         = grounding_words(answer, chunk_texts)
    r_sents         = grounding_sentences(answer, chunk_texts)

    print(f"Pregunta: {question}")
    print(f"\n--- Respuesta generada ---\n{answer}")
    print(f"\n--- Fuentes citadas ---")
    for i, c in enumerate(citations, 1):
        print(f"  [{i}] {c}")
    print(f"\n--- Chunks recuperados ---")
    for j, (_, r) in enumerate(chunks.iterrows()):
        print(f"  [{j+1}] score={r['score']:.3f} | {r['chunk_text'][:120]}...")
    print(f"\n--- Métricas ---")
    print(f"  Grounding words : {r_words['grounding_score']} — {r_words['label']}")
    print(f"  Grounding frases: {r_sents['grounding_score']} — {r_sents['label']}")

In [ ]:
# ============================================================
# Análisis cualitativo — 2 buenos + 2 malos
# ============================================================

print("=" * 70)
print("EJEMPLOS BUENOS")
print("=" * 70)

demo_question("What are the complications of Renal Artery Stenosis ?")
print("\n")
demo_question("What causes Pleurisy and Other Pleural Disorders ?")

print("\n")
print("=" * 70)
print("EJEMPLOS MALOS")
print("=" * 70)

demo_question("How many people are affected by spastic paraplegia type 11 ?")
print("\n")
demo_question("What to do for Viral Hepatitis: A through E and Beyond ?")

**Análisis cualitativo — Ejemplos buenos y malos**

---

**Ejemplos buenos:**

**1. "What are the complications of Renal Artery Stenosis?"**
El sistema recuperó 5 chunks relevantes con scores entre 0.565 y 0.626,
y el LLM generó una lista estructurada de complicaciones coherente con
el contexto. El grounding de frases fue perfecto (1.0) — todas las frases
generadas están respaldadas por los chunks recuperados. El grounding de
palabras (0.839) es alto aunque no perfecto, lo que indica que el LLM
usó algunos sinónimos o términos médicos relacionados no presentes
literalmente en los chunks.

Nota: la respuesta menciona "aneurysm" y "polycystic kidney disease"
que aparecen en los chunks de forma contextual — el LLM los incorporó
correctamente sin inventarlos.

**2. "What causes Pleurisy and Other Pleural Disorders?"**
Respuesta concisa y bien fundamentada. Grounding de frases perfecto (1.0)
a pesar de que el grounding de palabras es moderado (0.780). Esto ilustra
la diferencia entre ambas métricas: las frases completas están respaldadas
por el contexto pero el LLM parafraseó algunos términos. Los chunks
recuperados cubrieron bien las causas — viral infections, COPD, pulmonary
embolism, hemothorax — y el LLM las sintetizó correctamente.

---

**Ejemplos malos:**

**1. "How many people are affected by spastic paraplegia type 11?"**
El caso más claro de limitación del sistema. Los chunks recuperados
(scores 0.712-0.724) eran sobre spastic paraplegia en general —
prevalencia de type 4, prevalencia combinada de hereditary spastic
paraplegias — pero ninguno contenía el dato específico de "type 11".
El LLM respondió correctamente con "I don't know", activando el
mecanismo de grounding del system prompt, pero el grounding fue
(0.357/0.0) — la respuesta generada no comparte vocabulario con
el contexto porque simplemente admite no saber.

**2. "What to do for Viral Hepatitis: A through E and Beyond?"**
Pregunta demasiado amplia que requiere sintetizar información de múltiples
tipos de hepatitis. El LLM generó una lista de 9 recomendaciones generales,
de las cuales la mayoría no aparecen literalmente en los chunks recuperados
(grounding 0.454/0.2). Chunks con scores bajos (0.590-0.610) indican que
el retrieval no encontró un contexto suficientemente específico.

El LLM recurrió a conocimiento propio para completar la respuesta —
"Practice Good Hygiene", "Monitor Symptoms", "Inform Healthcare Providers"
son recomendaciones médicas genéricas no presentes en los chunks.
La respuesta además quedó truncada ("Consider Prevent...") por el límite
de `max_new_tokens=200`, lo que sugiere que este tipo de preguntas
requeriría un límite mayor.

---

**Conclusión:**

El sistema funciona bien con preguntas específicas y acotadas donde
la respuesta está concentrada en pocos chunks (complications, causes).
Falla en dos escenarios distintos:

- **Dato muy específico no recuperado**: el LLM responde "I don't know"
  correctamente pero no puede ayudar al usuario — limitación del retrieval.
- **Pregunta demasiado amplia**: el LLM sintetiza con conocimiento propio,
  aumentando el riesgo de alucinación y truncando la respuesta —
  limitación del `max_new_tokens` y del chunking con k=5.

# 12.Bonus — Benchmark comparativo entre arquitecturas RAG

El sistema ya implementa y evalúa, a lo largo de las secciones 8 y 9.1, **cuatro
arquitecturas de retrieval** distintas sobre el mismo corpus, los mismos
embeddings y el mismo LLM de generación:

- **Flat (FAISS `IndexFlatIP`)**: búsqueda vectorial exacta por similitud coseno. Baseline.
- **HNSW (FAISS `IndexHNSWFlat`)**: búsqueda vectorial aproximada, indexación jerárquica de grafos.
- **Hybrid (FAISS + BM25)**: combina similitud vectorial con score léxico BM25.
- **Reranker (FAISS + Cross-Encoder)**: recupera 20 candidatos por FAISS y los reordena con un cross-encoder antes de quedarse con el top-5.

Esta sección consolida esos resultados —ya calculados en `df_eval` (Recall@k /
Precision@k sobre 200 preguntas, sección 9.1) y `df_grounding` (grounding
sobre 3 preguntas, sección 8)— en una sola tabla comparativa por arquitectura,
con una conclusión explícita sobre cuál conviene usar en producción y por qué.


In [ ]:
# Consolidar Recall@k / Precision@k (df_eval, sección 9.1) y Grounding (df_grounding, sección 8)
# en una sola tabla comparativa por arquitectura (mode), promediando entre las 3 estrategias
# de chunking para aislar el efecto del modo de retrieval en sí.

bench_retrieval = (
    df_eval
    .groupby('mode')[['recall@k', 'precision@k']]
    .mean()
    .round(3)
)

bench_grounding = (
    df_grounding
    .groupby('mode')[['score_words', 'score_sentences']]
    .mean()
    .round(3)
)

df_benchmark = bench_retrieval.join(bench_grounding)
df_benchmark = df_benchmark.rename(columns={
    'recall@k'       : 'Recall@5 (avg)',
    'precision@k'    : 'Precision@5 (avg)',
    'score_words'    : 'Grounding palabras (avg)',
    'score_sentences': 'Grounding frases (avg)',
})

# Reordenar filas en un orden de lectura lógico
orden = ['flat', 'hnsw', 'hybrid', 'reranker']
df_benchmark = df_benchmark.reindex(orden)

print("Benchmark comparativo entre arquitecturas RAG (promedio sobre 3 estrategias de chunking):")
display(df_benchmark)


**Conclusión del benchmark**

| Arquitectura | Recall@5 | Precision@5 | Grounding palabras | Grounding frases | Lectura |
|---|---|---|---|---|---|
| **Flat** | 0.964 | 0.849 | 0.814 | 0.798 | Muy consistente en las 4 dimensiones — confirma ser el baseline más sólido, prácticamente a la par del reranker pero sin su costo computacional extra. |
| **HNSW** | 0.923 | 0.804 | 0.623 | 0.575 | El más bajo en las 4 métricas, y con una caída marcada en grounding (0.623 / 0.575 vs. 0.81 / 0.80 de flat) — la búsqueda aproximada sacrifica precisión real en este corpus, justo el costo que advertíamos: gana velocidad teórica a cambio de recuperar chunks menos útiles para fundamentar la respuesta. |
| **Hybrid** | 0.942 | 0.782 | 0.778 | 0.704 | Resultados intermedios — el aporte de BM25 ayuda algo respecto a HNSW puro, pero no supera a flat ni en retrieval ni en grounding; el ruido adicional del componente léxico se nota en Precision@5, el valor más bajo de la tabla (0.782). |
| **Reranker** | 0.966 | 0.850 | 0.803 | 0.767 | El mejor en Recall@5 y Precision@5 (por margen mínimo sobre flat), aunque ligeramente por debajo de flat en grounding por frases (0.767 vs 0.798) — el reordenamiento mejora qué se recupera, pero no necesariamente cómo se ensambla la respuesta final. |

**Recomendación para producción:** Flat y Reranker son prácticamente equivalentes en calidad (diferencias de centésimas), con Reranker apenas mejor en retrieval puro y Flat ligeramente mejor en grounding. Dado que Reranker añade un segundo modelo (cross-encoder) y más cómputo por consulta sin una mejora sustancial sobre Flat en este corpus de tamaño moderado, **Flat es la opción más eficiente para producción** — el mismo desempeño, menor latencia y menor complejidad de infraestructura. HNSW queda descartado para este caso de uso: la pérdida de grounding (caída de ~0.19 puntos frente a flat) no se justifica por una ganancia de velocidad que, en un corpus de este tamaño, es marginal.